# BERT-SSIN ABSA — UIT-VSFC (Tiếng Việt)

**Aspect-Category Sentiment Analysis** trên bộ dữ liệu UIT-VSFC (Đại học Công nghệ Thông tin - VNUHCM).

**Kiến trúc:** PhoBERT + Syncretic Info Network (SIN) + Semantic Guided Self-Attention (SGSA)

| Thành phần | Mô tả |
|---|---|
| **PhoBERT-base** | Pre-trained BERT cho tiếng Việt (VinAI Research) |
| **SyncreticInfoNetwork (SIN)** | GNN trên dependency tree, giúp nắm cấu trúc cú pháp |
| **SemanticGuidedAttention (SGSA)** | Multi-head attention dùng aspect-category vector làm query |
| **Focal Loss** | Xử lý class imbalance (neutral rất ít) |
| **Early stopping** | Dừng theo Macro-F1, tránh overfit |

**Nhãn:**
- `sentiment`: 0=Tiêu cực, 1=Trung tính, 2=Tích cực
- `topic` (aspect category): 0=Giảng viên, 1=Môn học, 2=Cơ sở vật chất, 3=Chung

## 1. Cài đặt thư viện

In [1]:
import subprocess, sys, os

# KHÔNG cài lại torch — Kaggle đã có sẵn PyTorch system-wide,
# cài đè trong cùng session không có tác dụng và gây conflict.
# Chỉ cài các thư viện còn thiếu.
pkgs = ['transformers==4.40.2', 'scikit-learn', 'underthesea', 'sentencepiece']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

import torch, transformers
print(f'✅ transformers {transformers.__version__}')
print(f'✅ torch        {torch.__version__}')
print(f'   CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU             : {torch.cuda.get_device_name(0)}')
    print(f'   CUDA (torch)    : {torch.version.cuda}')
try:
    import underthesea
    print(f'✅ underthesea {underthesea.__version__}')
except Exception as e:
    print(f'⚠️  underthesea: {e}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


✅ transformers 4.40.2
✅ torch        2.10.0+cu128
   CUDA available  : True
   GPU             : Tesla T4
   CUDA (torch)    : 12.8
✅ underthesea 9.4.0


## 2. Download dữ liệu UIT-VSFC

Download trực tiếp từ GitHub repo [`nguyenmaiductrong/absa-sota-survey`](https://github.com/nguyenmaiductrong/absa-sota-survey).

UIT-VSFC là dataset đánh giá sinh viên về các khía cạnh giảng dạy đại học:
- **Train:** 11,426 câu
- **Test:** 3,166 câu (gold test set)

In [2]:
import os, urllib.request

os.makedirs('/kaggle/working/uit-vsfc', exist_ok=True)

BASE = 'https://raw.githubusercontent.com/nguyenmaiductrong/absa-sota-survey/main/data/raw/uit-vsfc'
FILES = {
    'train.csv': f'{BASE}/train.csv',
    'test.csv':  f'{BASE}/test.csv',
}

for fname, url in FILES.items():
    dest = f'/kaggle/working/uit-vsfc/{fname}'
    if not os.path.exists(dest):
        print(f'⬇️  Downloading {fname}...')
        urllib.request.urlretrieve(url, dest)
        print(f'✅ {fname} saved')
    else:
        print(f'✅ {fname} already exists')

print('\n📁 Dữ liệu sẵn sàng!')

⬇️  Downloading train.csv...
✅ train.csv saved
⬇️  Downloading test.csv...
✅ test.csv saved

📁 Dữ liệu sẵn sàng!


## 3. Load & Khám phá dữ liệu

Dataset có 3 cột:
- `sentence`: câu tiếng Việt
- `sentiment`: nhãn cảm xúc (0=tiêu cực, 1=trung tính, 2=tích cực)
- `topic`: aspect category (0=giảng viên, 1=môn học, 2=cơ sở vật chất, 3=chung)

Mỗi câu được ghép với tên topic tương ứng làm **aspect text** để đưa vào mô hình dạng `[CLS] sentence [SEP] aspect_name [SEP]`.

In [3]:
import pandas as pd
import numpy as np
from collections import Counter

# Mapping topic → tên aspect tiếng Việt (dùng làm aspect text cho PhoBERT)
TOPIC_NAMES = {
    0: 'giảng viên',
    1: 'môn học',
    2: 'cơ sở vật chất',
    3: 'chung',
}
SENTIMENT_NAMES = {0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'}

train_df = pd.read_csv('/kaggle/working/uit-vsfc/train.csv')
test_df  = pd.read_csv('/kaggle/working/uit-vsfc/test.csv')

print(f'📊 Train: {len(train_df)} câu | Test: {len(test_df)} câu')
print()
print('=== Phân phối SENTIMENT (train) ===')
for k, v in sorted(train_df['sentiment'].value_counts().items()):
    print(f'  {SENTIMENT_NAMES[k]:12s} ({k}): {v:5d}  ({v/len(train_df)*100:.1f}%)')

print()
print('=== Phân phối TOPIC/ASPECT (train) ===')
for k, v in sorted(train_df['topic'].value_counts().items()):
    print(f'  {TOPIC_NAMES[k]:18s} ({k}): {v:5d}  ({v/len(train_df)*100:.1f}%)')

print()
print('=== Ví dụ mẫu ===')
for _, row in train_df.sample(5, random_state=42).iterrows():
    print(f'  [{TOPIC_NAMES[row.topic]}] "{row.sentence[:60]}..." → {SENTIMENT_NAMES[row.sentiment]}')

📊 Train: 11426 câu | Test: 3166 câu

=== Phân phối SENTIMENT (train) ===
  Tiêu cực     (0):  5325  (46.6%)
  Trung tính   (1):   458  (4.0%)
  Tích cực     (2):  5643  (49.4%)

=== Phân phối TOPIC/ASPECT (train) ===
  giảng viên         (0):  8166  (71.5%)
  môn học            (1):  2201  (19.3%)
  cơ sở vật chất     (2):   497  (4.3%)
  chung              (3):   562  (4.9%)

=== Ví dụ mẫu ===
  [giảng viên] "có khả năng truyền đạt tốt ...." → Tích cực
  [giảng viên] "khi có nhóm lên seminar thì phải có ít nhất ba câu hỏi thì m..." → Tiêu cực
  [giảng viên] "thầy giảng buồn ngủ , không nhiệt tình , bài tập thực hành k..." → Tiêu cực
  [giảng viên] "sữa lỗi cho sinh viên ...." → Tích cực
  [cơ sở vật chất] "phòng thực hành chưa có chất lượng ...." → Tiêu cực


## 4. Tiền xử lý & Dataset

**PhoBERT** được train trên văn bản tiếng Việt đã qua word segmentation.  
Dùng `underthesea.word_tokenize` để tách từ (ví dụ "giảng viên" → "giảng_viên").

**Dependency adj matrix:** Do underthesea dependency parse có thể không ổn định,
mình dùng **window-based adjacency** (window=3) aligned với PhoBERT subword tokens —
đủ để SIN nắm local syntactic context mà không phụ thuộc parser.

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

PHOBERT_MODEL = 'vinai/phobert-base'
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL)
print(f'✅ Tokenizer: {PHOBERT_MODEL}')
print(f'   Vocab size: {tokenizer.vocab_size:,}')

# ── Word segmentation tiếng Việt ──────────────────────────────────────────────
try:
    from underthesea import word_tokenize
    def vi_segment(text):
        """Segment Vietnamese text, return space-joined words."""
        return word_tokenize(text, format='text')
    # Test
    sample = 'thầy giảng bài rất nhiệt tình và dễ hiểu'
    print(f'\n✅ underthesea word_tokenize:')
    print(f'   Input : {sample}')
    print(f'   Output: {vi_segment(sample)}')
except Exception as e:
    print(f'⚠️  underthesea không khả dụng ({e}) → dùng raw text')
    def vi_segment(text):
        return text

# ── Window-based adj aligned với PhoBERT subword tokens ───────────────────────
def build_window_adj_phobert(text, tokenizer, max_len, window=3):
    """
    Build adjacency matrix aligned với PhoBERT subword tokens.
    Mỗi word (sau segmentation) map sang các subword tokens,
    edges là window-based giữa các words.
    """
    adj = np.eye(max_len, dtype=np.float32)
    try:
        segmented = vi_segment(text)
        words = segmented.split()

        # Map word index → list of subword token positions trong sequence
        word_to_tokens = {}
        cur_pos = 1  # 0 là <s> (CLS của PhoBERT dùng RoBERTa tokenizer)
        for wi, word in enumerate(words):
            sub = tokenizer.tokenize(word)
            n   = len(sub) if sub else 1
            end = min(cur_pos + n, max_len - 1)
            if cur_pos < max_len - 1:
                word_to_tokens[wi] = list(range(cur_pos, end))
            cur_pos += n
            if cur_pos >= max_len - 1:
                break

        # Window-based edges
        word_positions = list(word_to_tokens.keys())
        for idx, wi in enumerate(word_positions):
            for jdx in range(max(0, idx - window), min(len(word_positions), idx + window + 1)):
                wj = word_positions[jdx]
                for ti in word_to_tokens[wi]:
                    for tj in word_to_tokens[wj]:
                        if ti < max_len and tj < max_len:
                            adj[ti][tj] = adj[tj][ti] = 1.0
    except Exception:
        # Fallback: dense window trực tiếp trên token positions
        for i in range(min(max_len, 64)):
            for j in range(max(0, i - window), min(max_len, i + window + 1)):
                adj[i][j] = adj[j][i] = 1.0
    return adj

# ── Dataset ───────────────────────────────────────────────────────────────────
def df_to_samples(df):
    """Convert DataFrame → list of {text, aspect, polarity}."""
    samples = []
    for _, row in df.iterrows():
        samples.append({
            'text':     str(row['sentence']).strip().lower(),
            'aspect':   TOPIC_NAMES[int(row['topic'])],
            'polarity': int(row['sentiment']),
        })
    return samples

class PhoBertABSADataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=128):
        self.data = []
        for s in samples:
            # Segment tiếng Việt
            text_seg   = vi_segment(s['text'])
            aspect_seg = vi_segment(s['aspect'])

            enc = tokenizer(
                text_seg, aspect_seg,
                max_length=max_len, padding='max_length',
                truncation=True, return_tensors='pt'
            )
            input_ids      = enc['input_ids'].squeeze(0)
            attention_mask = enc['attention_mask'].squeeze(0)
            # PhoBERT (RoBERTa-based) không có token_type_ids →  zeros
            token_type_ids = torch.zeros(max_len, dtype=torch.long)

            # Aspect mask: tokens sau </s> đầu đến </s> cuối
            # PhoBERT dùng <s>=0, </s>=2
            sep_id  = tokenizer.sep_token_id  # </s> = 2
            sep_pos = (input_ids == sep_id).nonzero(as_tuple=True)[0]
            asp_start = sep_pos[0].item() + 1 if len(sep_pos) > 0 else 0
            asp_end   = sep_pos[1].item()     if len(sep_pos) > 1 else max_len
            asp_mask  = torch.zeros(max_len)
            asp_mask[asp_start:asp_end] = 1.0

            # Window-based adj aligned với PhoBERT
            adj = torch.tensor(
                build_window_adj_phobert(s['text'], tokenizer, max_len),
                dtype=torch.float
            )
            self.data.append({
                'input_ids':      input_ids,
                'attention_mask': attention_mask,
                'token_type_ids': token_type_ids,
                'asp_mask':       asp_mask,
                'adj':            adj,
                'polarity':       torch.tensor(s['polarity'], dtype=torch.long),
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

MAX_LEN    = 128
BATCH_SIZE = 16

train_samples = df_to_samples(train_df)
test_samples  = df_to_samples(test_df)
print(f'📊 Train: {len(train_samples)} | Test: {len(test_samples)}')

dist = Counter([s['polarity'] for s in train_samples])
print('\nPhân phối train:')
for k in sorted(dist):
    print(f'  {SENTIMENT_NAMES[k]}: {dist[k]}')

print('\n⏳ Build PhoBERT dataset (word segmentation ~3-5 phút)...')
train_dataset = PhoBertABSADataset(train_samples, tokenizer, MAX_LEN)
test_dataset  = PhoBertABSADataset(test_samples,  tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           num_workers=2, pin_memory=True)
print('✅ Dataset sẵn sàng!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer: vinai/phobert-base
   Vocab size: 64,000

✅ underthesea word_tokenize:
   Input : thầy giảng bài rất nhiệt tình và dễ hiểu
   Output: thầy giảng bài rất nhiệt_tình và dễ hiểu
📊 Train: 11426 | Test: 3166

Phân phối train:
  Tiêu cực: 5325
  Trung tính: 458
  Tích cực: 5643

⏳ Build PhoBERT dataset (word segmentation ~3-5 phút)...


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


✅ Dataset sẵn sàng!


## 5. Định nghĩa Model: PhoBERT-SSIN

Kiến trúc giữ nguyên từ BERT-SSIN v2, chỉ thay encoder backbone sang **PhoBERT-base**:

```
Input: [<s> sentence_tokens </s> aspect_tokens </s>]
         ↓
    PhoBERT encoder → seq_out (B, L, 768), cls_out (B, 768)
         ↓
    Linear proj → (B, L, 300)
         ↓
    SyncreticInfoNetwork (GNN, 2 layers) → h_syn (B, L, 300)
         ↓
    SemanticGuidedAttention (aspect vector làm query) → attn (B, 300)
         ↓
    Concat [attn, asp_proj(cls_out)] → (B, 600)
         ↓
    Classifier → logits (B, 3)
```

In [5]:
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import AutoModel

class SyncreticInfoNetwork(nn.Module):
    def __init__(self, dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.layers  = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_layers)])
        self.epsilon = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(num_layers)])
        self.drop    = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(dim)

    def forward(self, x, adj):
        h = x
        for i in range(len(self.layers)):
            agg = torch.bmm(adj, h)
            h   = F.relu(self.layers[i]((1 + self.epsilon[i]) * h + agg))
            h   = self.drop(h)
        return self.norm(h)


class SemanticGuidedAttention(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.H  = num_heads
        self.Dh = dim // num_heads
        self.W_q  = nn.Linear(dim, dim)
        self.W_k  = nn.Linear(dim, dim)
        self.W_v  = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, syn, asp_vec, mask=None):
        B, L, D = syn.shape
        q = self.W_q(asp_vec).unsqueeze(1).view(B, 1, self.H, self.Dh).transpose(1, 2)
        k = self.W_k(syn).view(B, L, self.H, self.Dh).transpose(1, 2)
        v = self.W_v(syn).view(B, L, self.H, self.Dh).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.Dh)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out  = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, 1, D)
        return self.norm(self.proj(out).squeeze(1) + asp_vec)


class PhoBertSSINClassifier(nn.Module):
    """
    PhoBERT + SyncreticInfoNetwork + SemanticGuidedAttention
    cho Aspect-Category Sentiment Analysis tiếng Việt (UIT-VSFC).
    """
    def __init__(self, phobert_model_name, hidden_dim=300, num_classes=3,
                 sin_layers=2, num_heads=4, dropout=0.3):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(phobert_model_name)
        bert_dim      = self.bert.config.hidden_size   # 768
        self.proj     = nn.Sequential(
            nn.Linear(bert_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )
        self.sin      = SyncreticInfoNetwork(hidden_dim, sin_layers, dropout)
        self.sgsa     = SemanticGuidedAttention(hidden_dim, num_heads, dropout)
        self.asp_proj = nn.Linear(bert_dim, hidden_dim)
        self.drop     = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, token_type_ids, adj, asp_mask):
        # PhoBERT (RoBERTa) không dùng token_type_ids → bỏ qua
        out     = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = out.last_hidden_state   # (B, L, 768)
        cls_out = out.last_hidden_state[:, 0, :]  # <s> token thay pooler

        h       = self.proj(seq_out)      # (B, L, hidden_dim)

        # Aspect vector: trung bình tokens của aspect category
        asp_len = asp_mask.sum(dim=1, keepdim=True).clamp(min=1)
        asp_vec = self.asp_proj(
            (seq_out * asp_mask.unsqueeze(-1)).sum(dim=1) / asp_len
        )   # (B, hidden_dim)

        # Normalise adj theo hàng
        adj_norm = adj / adj.sum(dim=-1, keepdim=True).clamp(min=1)
        h_syn    = self.sin(h, adj_norm)                       # (B, L, hidden_dim)

        attn     = self.sgsa(h_syn, asp_vec, attention_mask)   # (B, hidden_dim)
        feat     = torch.cat([attn, self.asp_proj(cls_out)], dim=-1)
        logits   = self.classifier(self.drop(feat))            # (B, 3)
        return logits


# ── Chọn device: thử CUDA, fallback CPU nếu kernel không tương thích ────────
# Lỗi 'no kernel image for device': PyTorch build không match GPU SM version.
# Giải pháp: detect lỗi sớm và dùng CPU — đúng hơn là crash khi training.

def _get_device():
    if not torch.cuda.is_available():
        print('ℹ️  CUDA không có sẵn → CPU')
        return torch.device('cpu')
    try:
        # Chạy một phép tính thực sự để kích hoạt CUDA kernel
        _t = torch.zeros(4, 4, device='cuda')
        _t = _t @ _t + _t
        del _t
        torch.cuda.synchronize()   # bắt lỗi async CUDA ngay tại đây
        torch.cuda.empty_cache()
        return torch.device('cuda')
    except Exception as _e:
        print(f'⚠️  CUDA kernel lỗi ({type(_e).__name__}): {_e}')
        print('   Nguyên nhân: PyTorch build không match SM version của GPU này.')
        print('   → Tự động dùng CPU. Training chậm hơn nhưng cho kết quả đúng.')
        # Tắt CUDA hoàn toàn để tránh các lỗi tiếp theo
        import os
        os.environ['CUDA_VISIBLE_DEVICES'] = ''
        return torch.device('cpu')

device = _get_device()
if device.type == 'cuda':
    print(f'🖥️  Device: cuda — {torch.cuda.get_device_name(0)}')
    print(f'   CUDA: {torch.version.cuda} | SM: {torch.cuda.get_device_capability(0)}')
else:
    print(f'🖥️  Device: cpu')

HIDDEN_DIM  = 300
NUM_CLASSES = 3

model = PhoBertSSINClassifier(PHOBERT_MODEL, HIDDEN_DIM, NUM_CLASSES).to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'📐 Total params: {n:,}')

🖥️  Device: cuda — Tesla T4
   CUDA: 12.8 | SM: (7, 5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

📐 Total params: 136,184,477


## 6. Focal Loss

Neutral class chỉ chiếm ~4% (458/11426) — imbalance rất nặng.
Focal Loss tập trung học các mẫu khó (neutral), tránh bị positive/negative dominant.

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t), \quad \gamma=2$$

In [6]:
class FocalLoss(nn.Module):
    """
    Focal Loss kết hợp class weights.
    gamma=2 theo Lin et al. (2017) - RetinaNet.
    """
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight    = weight
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=-1)
        prob     = torch.exp(log_prob)
        pt       = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_w  = (1 - pt) ** self.gamma
        ce       = F.nll_loss(log_prob, targets, weight=self.weight, reduction='none')
        loss     = focal_w * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


print('✅ FocalLoss defined')

✅ FocalLoss defined


## 7. Training

Hyperparameters:
- **BERT lr:** 2e-5 (fine-tune PhoBERT)
- **Head lr:** 1e-3 (SIN + SGSA + classifier)
- **Epochs:** tối đa 15, early stopping patience=4 theo Macro-F1
- **Scheduler:** linear warmup 10% → linear decay

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
import time

NUM_EPOCHS   = 15
PATIENCE     = 4

# Class weights cho Focal Loss
label_arr = np.array([s['polarity'] for s in train_samples])
weights   = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=label_arr)
print(f'⚖️  Class weights: Neg={weights[0]:.3f} | Neu={weights[1]:.3f} | Pos={weights[2]:.3f}')

criterion = FocalLoss(
    weight=torch.tensor(weights, dtype=torch.float).to(device),
    gamma=2.0
)

optimizer = torch.optim.AdamW([
    {'params': list(model.bert.parameters()), 'lr': 2e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'bert' not in n],
     'lr': 1e-3, 'weight_decay': 1e-4},
])
total_steps = len(train_loader) * NUM_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)


def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for b in loader:
            logits = model(
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                b['token_type_ids'].to(device),
                b['adj'].to(device),
                b['asp_mask'].to(device)
            )
            preds.extend(logits.argmax(-1).cpu().numpy())
            targets.extend(b['polarity'].numpy())
    acc = accuracy_score(targets, preds)
    f1  = f1_score(targets, preds, average='macro', zero_division=0)
    return acc, f1, targets, preds


history = {'loss': [], 'acc': [], 'f1': []}
best_acc, best_f1, best_state = 0.0, 0.0, None
patience_counter = 0

print(f'\n{"Epoch":>6} | {"Loss":>8} | {"Acc":>7} | {"F1":>7} | {"Time":>6}')
print('-' * 52)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for b in train_loader:
        optimizer.zero_grad()
        logits = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['token_type_ids'].to(device),
            b['adj'].to(device),
            b['asp_mask'].to(device)
        )
        loss = criterion(logits, b['polarity'].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    acc, f1, _, _ = evaluate(model, test_loader, device)
    history['loss'].append(avg_loss)
    history['acc'].append(acc)
    history['f1'].append(f1)

    if f1 > best_f1:
        best_f1, best_acc = f1, acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = ' ⭐'
    else:
        patience_counter += 1
        marker = ''

    elapsed = time.time() - t0
    print(f'{epoch:>6} | {avg_loss:>8.4f} | {acc:>7.4f} | {f1:>7.4f} | {elapsed:>5.1f}s{marker}')

    if patience_counter >= PATIENCE:
        print(f'\n⏹️  Early stopping tại epoch {epoch} (patience={PATIENCE})')
        break

print(f'\n🏆 Best Accuracy: {best_acc:.4f} | Best Macro-F1: {best_f1:.4f}')

⚖️  Class weights: Neg=0.715 | Neu=8.316 | Pos=0.675

 Epoch |     Loss |     Acc |      F1 |   Time
----------------------------------------------------
     1 |   0.3420 |  0.9277 |  0.8252 | 310.1s ⭐


## 8. Đánh giá trên tập Test Gold

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

model.load_state_dict(best_state)
acc, f1, targets, preds = evaluate(model, test_loader, device)

print('=' * 65)
print('  PhoBERT-SSIN — UIT-VSFC Vietnamese ABSA')
print('=' * 65)
print(classification_report(
    targets, preds,
    target_names=['Tiêu cực', 'Trung tính', 'Tích cực'],
    digits=4
))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['loss'], 'b-o', markersize=4)
axes[0].set_title('Training Loss (Focal Loss)')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)

axes[1].plot(history['acc'], 'g-o', markersize=4, label='Accuracy')
axes[1].plot(history['f1'],  'r-s', markersize=4, label='Macro-F1')
axes[1].set_title('Test Accuracy & Macro-F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

cm = confusion_matrix(targets, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Tiêu cực', 'Trung tính', 'Tích cực'],
            yticklabels=['Tiêu cực', 'Trung tính', 'Tích cực'])
axes[2].set_title('Confusion Matrix')
axes[2].set_ylabel('True')
axes[2].set_xlabel('Predicted')
plt.xticks(rotation=30)
plt.yticks(rotation=0)

plt.suptitle(
    f'PhoBERT-SSIN | UIT-VSFC | Acc={acc:.4f} | Macro-F1={f1:.4f}',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/phobert_ssin_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Lưu Model

In [ ]:
torch.save(
    {'model_state': best_state, 'best_acc': best_acc, 'best_f1': best_f1,
     'phobert_model': PHOBERT_MODEL, 'hidden_dim': HIDDEN_DIM,
     'topic_names': TOPIC_NAMES, 'sentiment_names': SENTIMENT_NAMES},
    '/kaggle/working/phobert_ssin_best.pt'
)
print('✅ Saved: phobert_ssin_best.pt')
print(f'   Best Acc  : {best_acc:.4f}')
print(f'   Best F1   : {best_f1:.4f}')

## 10. Demo Inference

Test với các câu tiếng Việt thực tế từ domain giảng dạy đại học.

In [ ]:
def predict(text, topic_id, model, tokenizer, device, max_len=128):
    """
    Dự đoán sentiment cho một câu + topic category.
    topic_id: 0=giảng viên, 1=môn học, 2=cơ sở vật chất, 3=chung
    """
    model.eval()
    aspect   = TOPIC_NAMES[topic_id]
    text_seg = vi_segment(text.lower())
    asp_seg  = vi_segment(aspect)

    enc = tokenizer(
        text_seg, asp_seg,
        max_length=max_len, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    ids = enc['input_ids'].to(device)
    sep_id  = tokenizer.sep_token_id
    sep_pos = (ids[0] == sep_id).nonzero(as_tuple=True)[0]
    asp_mask = torch.zeros(1, max_len).to(device)
    if len(sep_pos) >= 2:
        asp_mask[0, sep_pos[0] + 1:sep_pos[1]] = 1.0

    adj = torch.tensor(
        [build_window_adj_phobert(text.lower(), tokenizer, max_len)],
        dtype=torch.float
    ).to(device)
    token_type_ids = torch.zeros(1, max_len, dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(ids, enc['attention_mask'].to(device),
                       token_type_ids, adj, asp_mask)
        prob = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        pred = logits.argmax(-1).item()
    return ['Tiêu cực 😤', 'Trung tính 😐', 'Tích cực 😊'][pred], prob


print('🔍 Demo Inference — UIT-VSFC')
print('=' * 70)
test_cases = [
    ('thầy giảng bài rất nhiệt tình và dễ hiểu',           0),
    ('giảng viên thường xuyên đến muộn và thiếu chuẩn bị', 0),
    ('môn học này khá bổ ích và thực tế',                   1),
    ('nội dung môn học quá lý thuyết và nhàm chán',         1),
    ('phòng học rộng rãi và có điều hòa mát',               2),
    ('wifi trường rất chậm ảnh hưởng việc học',             2),
    ('tôi hài lòng với tất cả mọi thứ',                     3),
    ('không có ý kiến gì thêm',                             3),
]

for text, topic_id in test_cases:
    label, probs = predict(text, topic_id, model, tokenizer, device)
    print(f'Câu   : "{text}"')
    print(f'Aspect: [{TOPIC_NAMES[topic_id]}]  →  {label}')
    print(f'Probs : Tiêu cực={probs[0]:.3f} | Trung tính={probs[1]:.3f} | Tích cực={probs[2]:.3f}')
    print('-' * 70)